# Stateful EQX Models (BatchNorm)

This tutorial shows how to train an Equinox model that contains **mutable state** — such as `eqx.nn.BatchNorm` running statistics — using `EQXTrainer`.

Equinox uses a functional approach: stateful layers (anything with `eqx.nn.State`) return updated state from the forward pass. `EQXTrainer` threads this state through each training step and persists it across epochs via the `train_state=` argument to `.train()`.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import equinox as eqx

from trainax import EQXTrainer, JaxLoader, StepOutput, ValStepOutput, EpochLogger

## 1. Data

In [ ]:
np.random.seed(0)
x = np.random.randn(500, 16).astype(np.float32)
y = (x @ np.random.randn(16, 1)).astype(np.float32).squeeze(-1)

trainloader = JaxLoader({"x": x[:400], "y": y[:400]}, batch_size=32)
valloader   = JaxLoader({"x": x[400:], "y": y[400:]}, batch_size=32)

print(f"Train batches: {len(trainloader)}, Val batches: {len(valloader)}")

## 2. Model with BatchNorm

We use `eqx.nn.make_with_state` to separate the model pytree from its initial state (running mean/var for BatchNorm).

In [ ]:
class BNModel(eqx.Module):
    linear1: eqx.nn.Linear
    bn: eqx.nn.BatchNorm
    linear2: eqx.nn.Linear

    def __init__(self, key: jax.Array):
        k1, k2 = jax.random.split(key)
        self.linear1 = eqx.nn.Linear(16, 32, key=k1)
        self.bn = eqx.nn.BatchNorm(32, axis_name="batch")
        self.linear2 = eqx.nn.Linear(32, 1, key=k2)

    def __call__(self, x: jax.Array, state: eqx.nn.State) -> tuple[jax.Array, eqx.nn.State]:
        x = self.linear1(x)
        x, state = self.bn(x, state)
        x = jax.nn.relu(x)
        return self.linear2(x).squeeze(-1), state


model, state = eqx.nn.make_with_state(BNModel)(jax.random.key(0))
print("Model created. State type:", type(state))

## 3. Step functions

When a model has state, the **StateStepFun** signature applies:
```
(model, batch, state) -> StepOutput
```
The trainer detects state is present by checking `train_state != None` and passes it through. The step function must return a `StepOutput` with `out.state` set to the updated state.

In [ ]:
def train_step(model, batch, state):
    def loss_fn(m):
        # vmap over the batch axis; BatchNorm needs axis_name="batch"
        yhat, new_state = jax.vmap(
            m, axis_name="batch", in_axes=(0, None), out_axes=(0, None)
        )(batch["x"], state)
        loss = jnp.mean((yhat - batch["y"]) ** 2)
        return loss, (yhat, new_state)

    (loss, (yhat, new_state)), grads = eqx.filter_value_and_grad(
        loss_fn, has_aux=True
    )(model)
    return StepOutput(
        loss=loss,
        y=batch["y"],
        yhat=yhat,
        gradients=grads,
        state=new_state,  # <- updated BatchNorm statistics
    )


def val_step(model, batch):
    # EQXTrainer wraps the model with eqx.Partial(model, state=state) before
    # calling inference mode, so val_step receives the partial and sees no state arg.
    yhat = jax.vmap(model, axis_name="batch")(batch["x"])
    loss = jnp.mean((yhat - batch["y"]) ** 2)
    return ValStepOutput(loss=loss, y=batch["y"], yhat=yhat)

## 4. Train

Pass `train_state=state` to `.train()`. The trainer will thread it through every step and keep the latest version across epochs.

In [ ]:
trainer = EQXTrainer(
    n_epochs=10,
    callbacks=[EpochLogger("logger")],
    val_every=2,
    use_rich=False,
)

trained_model, trained_state = trainer.train(
    model,
    optax.adam(1e-3),
    train_step,
    trainloader,
    val_step=val_step,
    valloader=valloader,
    train_state=state,  # <- pass initial state here
)

print("Training complete. Trained state type:", type(trained_state))

## 5. Inference

At inference time, set the model to inference mode and pass the final state:

In [ ]:
inference_model = eqx.nn.inference_mode(trained_model)

x_test = np.random.randn(16, 16).astype(np.float32)
preds, _ = jax.vmap(
    inference_model, axis_name="batch", in_axes=(0, None), out_axes=(0, None)
)(x_test, trained_state)

print("Predictions shape:", preds.shape)